# Meta Exploration — Immortal Bracket

Quick analysis to understand the current patch meta at 7k+ MMR.

**Requirements:** Run `dotaengineer ingest meta` first to populate DuckDB.

In [ ]:
import sys
sys.path.insert(0, '../src')

import duckdb
import polars as pl
import plotly.express as px

from dotaengineer.config import settings
from dotaengineer.analysis.meta import MetaAnalyzer
from dotaengineer.analysis.draft import DraftAnalyzer
from dotaengineer.analysis.player_performance import PlayerPerformanceAnalyzer

pl.Config.set_tbl_rows(30)

## 1. Tier list — Immortal bracket

In [ ]:
meta = MetaAnalyzer()
tiers = meta.tier_list()
tiers.head(30)

In [ ]:
fig = px.scatter(
    tiers.to_pandas(),
    x='pick_rate', y='win_rate',
    color='tier', size='contest_rate',
    hover_name='hero_name',
    color_discrete_map={'S': '#FFD700', 'A': '#00C853', 'B': '#2196F3', 'C': '#9E9E9E'},
    title='Win Rate vs Pick Rate — Immortal',
)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', opacity=0.4)
fig.show()

## 2. Sleeper picks — High WR, Low PR

In [ ]:
meta.sleeper_picks()

## 3. Role meta

In [ ]:
meta.role_meta()

## 4. Draft tool — Counter-pick analysis

Replace hero IDs with the actual enemy picks in your lobby.

In [ ]:
draft = DraftAnalyzer()

# Example: enemy picked Invoker (74), Pudge (14), Phantom Assassin (44)
enemy = [74, 14, 44]
my_team = []  # heroes already picked on my side
my_pool = []  # restrict to your hero pool (empty = all heroes)

best = draft.get_best_pick(my_team=my_team, enemy_team=enemy, pool=my_pool or None)
best

In [ ]:
# Full draft evaluation
# radiant = my team, dire = enemy team
draft.analyze_draft(radiant=[1, 2, 3, 4, 5], dire=[6, 7, 8, 9, 10])

## 5. Personal performance

Run `dotaengineer ingest player` first.

In [ ]:
perf = PlayerPerformanceAnalyzer()

print("=== Hero Pool ===")
perf.hero_pool_summary()

In [ ]:
print("=== Worst heroes vs meta ===")
perf.worst_heroes()

In [ ]:
print("=== Farm efficiency ===")
perf.farm_efficiency()

In [ ]:
print("=== Win rate by game duration ===")
splits = perf.win_loss_by_duration()
splits

In [ ]:
# Rolling form — last 50 games
trend = perf.recent_trend(last_n=50)
fig2 = px.line(
    trend.to_pandas(),
    x='start_time', y='rolling_10_wr',
    title='Rolling 10-game Win Rate (recent form)',
)
fig2.add_hline(y=0.5, line_dash='dash', line_color='red', opacity=0.4)
fig2.show()